# Data structures

| Feature | Series | DataFrame |
|---------|--------|-----------|
| **Dimensions** | 1D array | 2D matrix |
| **Row Index** | Numerical (0, 1, 2, ...) | String or float |
| **Column Index** | String or float| String or float |
| **Data Type** | Single type | Multiple types per column |
| **Example** | `[10, 20, 30]` | `{'Name': [...], 'Age': [...]}` |

- **categorical data**
    - Pandas keeps the unique values in a separate array and then stores the values as integers that refer to the individual values.
- **datetime**
    - Generally want to use datetime64[ns] for Pandas 1.x

In [2]:
from IPython.display import display
import pandas as pd
import numpy as np
states = ['CA', np.nan, 'NY', 'TX']
display(pd.Series(states, dtype='category'))
print('category data type ignore the NAN type')
display(pd.Series(states).astype('category'))

0     CA
1    NaN
2     NY
3     TX
dtype: category
Categories (3, object): ['CA', 'NY', 'TX']

category data type ignore the NAN type


0     CA
1    NaN
2     NY
3     TX
dtype: category
Categories (3, object): ['CA', 'NY', 'TX']

### datetime example

method 1: use `dtype='datetime64[ns]'`

In [ ]:
import datetime as dt
dt_list = [dt.datetime(2020, 1, 1, 4, 30 ), dt.datetime(2020, 1, 2),\
    dt.datetime(2020, 1, 3)]

string_dates = ['2020-01-01 04:30:00', '2020-01-02 00:00:00', \
    '2020-01-03 00:00:00']
string_dates_missing = ['2020-01-01 4:30', None, '2020-01-03']
epoch_dates = [1577836800, 1577923200, 1578009600] # seconds away
pd.Series(dt_list)

0   2020-01-01 04:30:00
1   2020-01-02 00:00:00
2   2020-01-03 00:00:00
dtype: datetime64[ns]

In [12]:
string_dates_missing2=pd.Series(string_dates_missing, dtype='datetime64[ns]')
display(string_dates_missing2)

# use dt method to convert between different time formates
pd.to_datetime(string_dates_missing2).dt.normalize()

0   2020-01-01 04:30:00
1                   NaT
2   2020-01-03 00:00:00
dtype: datetime64[ns]

0   2020-01-01
1          NaT
2   2020-01-03
dtype: datetime64[ns]

In [13]:
pd.Series(epoch_dates, dtype='datetime64[s]') # starting from 1970.

0   2020-01-01
1   2020-01-02
2   2020-01-03
dtype: datetime64[s]

method 2: use `to_datetime()`

> Note: `dtype: datetime64[ns]` means the internal precision is nanoseconds, not that nanoseconds must be shown.

In [8]:
pd.Series(string_dates, dtype='datetime64[ns]')

0   2020-01-01 04:30:00
1   2020-01-02 00:00:00
2   2020-01-03 00:00:00
dtype: datetime64[ns]

In [7]:
pd.to_datetime(pd.Series([i[0:10] for i in string_dates]))

0   2020-01-01
1   2020-01-02
2   2020-01-03
dtype: datetime64[ns]

In [9]:
pd.Series([i[0:10] for i in string_dates], dtype='datetime64[ns]')

0   2020-01-01
1   2020-01-02
2   2020-01-03
dtype: datetime64[ns]

# Intro to series

Pandas series can have indices by string, categorical variables, or date time.
| Function | Description |
|---|---|
| pd.Series(data=None, index=None, dtype=None, name=None, copy=False) | Create a Series from data (sequence, dict, or scalar). |
| s.index | Access index of the Series. |
| s.astype(dtype, errors='raise') | Cast Series to dtype. Use errors='ignore' to return the original on failure. |
| s[boolean_array] | Select values where boolean_array is True. |
| s.cat.ordered | Check whether a categorical Series is ordered. |
| s.cat.reorder_categories(new_categories, ordered=False) | Reorder/replace categories; new_categories must include all existing categories. |
| s.cat.categories | Return categories of a categorical Series. |
| s.cat.add_categories(new_categories) | Add new categories to a categorical Series. |

In [14]:
nan_series = pd.Series([2, np.nan, 10], index=['Ono', 'Clapton', 'Misa'])
nan_series.index

Index(['Ono', 'Clapton', 'Misa'], dtype='object')

In [15]:
display(nan_series['Ono']) # np.nan is a float64!
display(nan_series.count()) # for non-na values
display(nan_series.size)    # identical to len()
display(len(nan_series))
display(nan_series[nan_series>3].index) # can use math and mask as numpy

np.float64(2.0)

np.int64(2)

3

3

Index(['Misa'], dtype='object')

In [ ]:
nan_series>3 # vectorizing 3

Ono        False
Clapton    False
Misa        True
dtype: bool

### filtering based on ordered categories

In [17]:
s = pd.Series(['s', 'm', 'l', 's', 'm', 's'], dtype='category')
display(s.cat.ordered)

s2 = pd.Series(['m', 'l', 'xs', 's', 'xl'])
# create a type with the CategoricalDtype constructor and the
# appropriate parameters to convert a non-categorical series to an ordered category
size_type = pd.CategoricalDtype(categories=['s','m','l'], ordered=True)  

s3 = s2.astype(size_type)      
display(s3)        

# once ordered, we can compare
display(s3>'m')

# add addition categories to an existing categorical series and set the order
# remember to use cat b/c we are modifying the attribute cat
s.cat.add_categories(['xl', 'xs']) \
    .cat.reorder_categories(['xs','s','m','l', 'xl'], ordered=True)

False

0      m
1      l
2    NaN
3      s
4    NaN
dtype: category
Categories (3, object): ['s' < 'm' < 'l']

0    False
1     True
2    False
3    False
4    False
dtype: bool

0    s
1    m
2    l
3    s
4    m
5    s
dtype: category
Categories (5, object): ['xs' < 's' < 'm' < 'l' < 'xl']

# Intermediate series

## dunker method "+" as ".add()"

| Method / Function | Operator | Description |
|---|---:|---|
| s.add(s2) | s + s2 | Adds series |
| s.sub(s2) | s - s2 | Subtracts series |
| s.mul(s2) / s.multiply(s2) | s * s2 | Multiplies series |
| s.div(s2) / s.truediv(s2) | s / s2 | Divides series |
| s.mod(s2) | s % s2 | Modulo of series division |
| s.floordiv(s2) | s // s2 | Floor divides series |
| s.pow(s2) | s ** s2 | Exponential power of series |
| s.eq(s2) | s == s2 | Elementwise equals |
| s.ne(s2) | s != s2 | Elementwise not equals |
| s.gt(s2) | s > s2 | Elementwise greater than |
| s.ge(s2) | s >= s2 | Elementwise greater than or equals |
| s.lt(s2) | s < s2 | Elementwise less than |
| s.le(s2) | s <= s2 | Elementwise less than or equals |
| np.invert(s) | ~s | Elementwise inversion of boolean series (no pandas method) |
| np.logical_and(s, s2) | s & s2 | Elementwise logical AND of boolean series (no pandas method) |
| np.logical_or(s, s2) | s \| s2 | Elementwise logical OR of boolean series (no pandas method) |

### index alignment
Pandas ensures that the index of the two series is aligned before performing math operations.


In [ ]:
s1 = pd.Series([10, 20, 30], index=[1,2,2])
s2 = pd.Series([    35, 44, 53], index=[2,2,4], name='s2')

# adding aligns index
display(s1+s2)

# `add` manages missing values
display(s1.add(s2, fill_value=0))

# adding ignoring index alignment
display(s1.reset_index())
display(s1.reset_index(drop=True)+s2.reset_index(drop=True)) 

1     NaN
2    55.0
2    64.0
2    65.0
2    74.0
4     NaN
dtype: float64

1    10.0
2    55.0
2    64.0
2    65.0
2    74.0
4    53.0
dtype: float64

,index,0
0,1,10
1,2,20
2,2,30


0    45
1    64
2    83
dtype: int64

## aggregation

| Method / Property | Description |
|---|---|
| s.agg(func=None, axis=0, \*args, \*\*kwargs) | Scalar if func is single aggregation; Series if func is list. |
| s.all(axis=0, bool_only=None, skipna=True, level=None) | True if every value is truthy. |
| s.any(axis=0, bool_only=None, skipna=True, level=None) | True if any value is truthy. |
| s.autocorr(lag=1) | Pearson correlation between s and shifted s (lag). |
| s.corr(other, method='pearson') | Correlation with other series ('pearson','spearman','kendall', or callable). |
| s.cov(other, min_periods=None) | Covariance with other series. |
| s.max(axis=None, skipna=None, level=None, numeric_only=None) | Maximum value. |
| s.min(axis=None, skipna=None, level=None, numeric_only=None) | Minimum value. |
| s.mean(axis=None, skipna=None, level=None, numeric_only=None) | Mean value. |
| s.median(axis=None, skipna=None, level=None, numeric_only=None) | Median value. |
| s.prod(axis=None, skipna=None, level=None, numeric_only=None, min_count=0) | Product of values. |
| s.quantile(q=0.5, interpolation='linear') | Quantile(s); returns Series if q is list. |
| s.sem(axis=None, skipna=None, level=None, ddof=1, numeric_only=None) | Unbiased standard error of the mean. |
| s.std(axis=None, skipna=None, level=None, ddof=1, numeric_only=None) | Sample standard deviation. |
| s.var(axis=None, skipna=None, level=None, ddof=1, numeric_only=None) | Unbiased variance. |
| s.skew(axis=None, skipna=None, level=None, numeric_only=None) | Unbiased skew. |
| s.kurtosis(axis=None, skipna=None, level=None, numeric_only=None) | Unbiased kurtosis. |
| s.nunique(dropna=True) | Count of unique items. |
| s.count(level=None) | Count of non-missing items. |
| s.size | Number of items in series (property). |
| s.is_unique | True if all values are unique (property). |
| s.is_monotonic / s.is_monotonic_increasing / s.is_monotonic_decreasing | Monotonicity checks (increasing/decreasing). |

Also `.kurt()` for Excess kurtosis (0 = normal distribution),`.mad()` for mean absolute deviation, `.sum()`, `ndim` returns the trivial property of the series.

| Other Methods | Description |
|---|---|
|`dtype` / `.dtypes` | Type of the series. |
| `empty` | True if series has no values. |
| `.hasnans` | True if series contains missing values. |
| `.idxmax()`| Index label of the maximum value. |
| `.idxmin()` | Index label of the minimum value. |
| `.count()` | Count of non-missing values. |

In [55]:
pd.Series(np.arange(1, 101, 1)).agg(['mean', 'var', 'max'])

mean     50.500000
var     841.666667
max     100.000000
dtype: float64

> s.var() uses the sample (unbiased) variance estimator by default (ddof=1).

> Formula: var = sum((x - x̄)^2) / (n - ddof) where ddof=1 by default.

In [ ]:
miles=pd.Series(np.arange(11))
miles.quantile([0.9, 0.6, 0.1]) 

0.9    9.0
0.6    6.0
0.1    1.0
dtype: float64

## converting types
| Method / Function | Description |
|---|---|
| s.convert_dtypes(infer_objects=True, convert_string=True, convert_integer=True, convert_boolean=True, convert_floating=True) | Convert types to appropriate pandas 1.x types that support NA. Does not downcast integer/float sizes. |
| s.astype(dtype, copy=True, errors='raise') | Cast Series to a specific dtype. Use errors='ignore' to return the original on error. |
| pd.to_datetime(arg, errors='raise', dayfirst=False, yearfirst=False, utc=None, format=None, exact=True, unit=None, infer_datetime_format=False, origin='unix', cache=True) | Parse arg (e.g., Series) into datetimes. Use format to specify strftime parsing. |
| s.to_numpy(dtype=None, copy=False, na_value=object, **kwargs) | Convert the Series to a NumPy array. |
| s.values | Shortcut to convert the Series to a NumPy array. |
| s.to_frame() | Return a DataFrame representation of the Series. |
| pd.CategoricalDtype(categories=None, ordered=False) | Construct a CategoricalDtype for categorical data. |

In [56]:
values = pd.Series(sorted(set([1, 10, 2, 3, 9, 8, 7, 6])))
values.astype('category').cat.as_ordered()

0     1
1     2
2     3
3     6
4     7
5     8
6     9
7    10
dtype: category
Categories (8, int64): [1 < 2 < 3 < 6 < 7 < 8 < 9 < 10]

## manipulating values & keeping index

In [45]:
# sample data
import numpy as np
import pandas as pd
vals = [10, 12, np.nan, 11, 10, 1000, 12, 13, 10, np.nan]  # duplicates (10,12), outlier (1000), missing (np.nan)
index = [f"row_{i}" for i in range(1, len(vals) + 1)]      # string index
s = pd.Series(vals, index=index, name="sample_series")
print(s)

row_1       10.0
row_2       12.0
row_3        NaN
row_4       11.0
row_5       10.0
row_6     1000.0
row_7       12.0
row_8       13.0
row_9       10.0
row_10       NaN
Name: sample_series, dtype: float64


### missing value and interpolation
| Method / Function | Description |
|---|---|
|s.isna() | finding np.NaN|
| s.fillna(value=None, method=None, axis=None, inplace=False, limit=None, downcast=None) | Fill missing values with scalar/dict/Series/DataFrame. |
|s.ffill(), s.bfill(), s.dropna()| |
| s.interpolate(method='linear', axis=0, limit=None, inplace=False, limit_direction=None, limit_area=None, downcast=None, **kwargs) | Interpolate missing values. method can be 'linear', 'time', etc. |


In [46]:
s.isna().sum() # total missing values
s.fillna(value=pd.Series([1,2])) # the don't modify original series
s.fillna(value=1)
s.fillna(value={'row_3': 1, 'row_10': 2})
s.dropna()
print(s) 

row_1       10.0
row_2       12.0
row_3        NaN
row_4       11.0
row_5       10.0
row_6     1000.0
row_7       12.0
row_8       13.0
row_9       10.0
row_10       NaN
Name: sample_series, dtype: float64


In [18]:
vals = [10, 12, np.nan, 11, 10, 1000, 12, 13, 10, np.nan]  # duplicates (10,12), outlier (1000), missing (np.nan)
index = [f"row_{i}" for i in range(1, len(vals) + 1)]      # string index
s = pd.Series(vals, index=index, name="sample_series")
s.loc[s.isna()] = [1,2] # modifes the original series
print(s)

row_1       10.0
row_2       12.0
row_3        1.0
row_4       11.0
row_5       10.0
row_6     1000.0
row_7       12.0
row_8       13.0
row_9       10.0
row_10       2.0
Name: sample_series, dtype: float64


### Clipping outliers
| Method / Function | Description |
|---|---|
| s.clip(lower=None, upper=None, axis=None, inplace=False, *args, **kwargs) | Clip values to the provided lower and/or upper bounds. |

In [47]:
vals = [10, 12, np.nan, 11, 10, 1000, 12, 13, 10, np.nan]  # duplicates (10,12), outlier (1000), missing (np.nan)
index = [f"row_{i}" for i in range(1, len(vals) + 1)]      # string index
s = pd.Series(vals, index=index, name="sample_series")
s.clip(lower=s.quantile(0.2), upper=s.quantile(0.8))

row_1     10.0
row_2     12.0
row_3      NaN
row_4     11.0
row_5     10.0
row_6     12.6
row_7     12.0
row_8     12.6
row_9     10.0
row_10     NaN
Name: sample_series, dtype: float64

### Sorting values/index
| Method / Function | Description |
|---|---|
| s.sort_values(axis=0, ascending=True, inplace=False, kind='quicksort', na_position='last', ignore_index=False, key=None) | Return Series sorted by values. kind options include 'quicksort','mergesort' (stable), 'heapsort'. |
| s.sort_index(axis=0, level=None, ascending=True, inplace=False, kind='quicksort', na_position='last', sort_remaining=True, ignore_index=False, key=None) | Return Series with index sorted. kind and na_position as in sort_values. |

### Dropping duplicates data
| Method / Function | Description |
|---|---|
| s.drop_duplicates(keep='first', inplace=False) | Drop duplicate values. keep may be 'first', 'last', or False (remove all duplicates). |

In [48]:
s.drop_duplicates(keep='first')

row_1      10.0
row_2      12.0
row_3       NaN
row_4      11.0
row_6    1000.0
row_8      13.0
Name: sample_series, dtype: float64

### Ranking data
| Method / Function | Description |
|---|---|
| s.rank(axis=0, method='average', numeric_only=None, na_option='keep', ascending=True, pct=False) | Return numerical ranks. method handles ties: 'average','min','max','first','dense'. na_option: 'keep','top','bottom'. pct=True returns ranks as percentages. |

In [ ]:
s.rank(method='dense')  # same value, same rank 1, 2, 3,...

row_1     1.0
row_2     3.0
row_3     NaN
row_4     2.0
row_5     1.0
row_6     5.0
row_7     3.0
row_8     4.0
row_9     1.0
row_10    NaN
Name: sample_series, dtype: float64

In [ ]:
s.rank(method='first') # same value, diff rank 1, 2, 3,...

row_1     1.0
row_2     5.0
row_3     NaN
row_4     4.0
row_5     2.0
row_6     8.0
row_7     6.0
row_8     7.0
row_9     3.0
row_10    NaN
Name: sample_series, dtype: float64

### Replacing data
> note: `str.replace(old, new)` in python

| Method / Function | Description |
|---|---|
| s.replace(to_replace=None, value=None, inplace=False, limit=None, regex=False, method='pad') | Replace values. to_replace can be scalar, list, regex, or dict mapping old->new. If list, value must match length. Supports inplace, limit, regex and fill methods. |

### Binning data
| Method / Function | Description |
|---|---|
| pd.cut(x, bins, right=True, labels=None, retbins=False, precision=3, include_lowest=False, duplicates='raise', ordered=True) | Bin values into equal-width bins (if bins is int) or into specified edges (if bins is list). right controls interval closure. labels names bins. Out-of-range -> NaN. |
| pd.qcut(x, q, labels=None, retbins=False, precision=3, duplicates='raise') | Bin values into quantile-based equal-sized bins (q can be int or list of quantile edges). Returns categorical bins; out-of-range -> NaN. |

### .apply and .where
`.apply`
- apply a function to an entire series using Numpy functions
    - OK
- element-wise to every value
    - slow (see below)

`.where`
- keep the series where the boolean array is true and replace values where it is false

| Method / Function | Description |
|---|---|
| s.apply(func, convert_dtype=True, args=(), **kwds) | Apply a NumPy or Python function to the Series (elementwise or returns Series/DataFrame). args and kwds passed to func. |
| s.where(cond, other=nan, inplace=False, axis=None, level=None, errors='raise', try_cast=False) | Keep values where cond is True, otherwise use other. cond can be boolean Series/list or callable returning booleans. |
| s.case_when(caselist) | Simulate if/then using list of (boolean_array, result) tuples. |

> Note: `.apply()` has broader functionality than `.map()`, which is used for simply substitution based on a value-to-value mapping. Unmatched dict keys become NaN.

In [ ]:
import pandas as pd
se=pd.Series([1, 2, 3, 4, 5])
se.map({1:0, 3:0})


0    0.0
1    NaN
2    0.0
3    NaN
4    NaN
dtype: float64

In [12]:
se.map(lambda x: x**2 if x%2==0 else x) # not that I need else in the lambda function

0     1
1     4
2     3
3    16
4     5
dtype: int64

>> Note: **else** is not needed in the list generator `[x**2 for x in se if x % 2 == 0]`

In [44]:
%%timeit
vals = [10, 12, np.nan, 11, 10, 1000, 12, 13, 10, np.nan]  # duplicates (10,12), outlier (1000), missing (np.nan)
vals2=[]
for i in range(100):
    vals2.extend(vals)

index = [f"row_{i}" for i in range(1, len(vals2) + 1)]      # string index
s = pd.Series(vals2, index=index, name="sample_series")

vals2.sort
lowest2=vals2[:2]
def change_lowest2(val):
    if val in lowest2:
        return val
    return 20
s.apply(change_lowest2)


505 µs ± 3.65 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [41]:
%%timeit
vals = [10, 12, np.nan, 11, 10, 1000, 12, 13, 10, np.nan]  # duplicates (10,12), outlier (1000), missing (np.nan)
vals2=[]
for i in range(100):
    vals2.extend(vals)

index = [f"row_{i}" for i in range(1, len(vals2) + 1)]      # string index
s = pd.Series(vals2, index=index, name="sample_series")

vals2.sort
lowest2=vals2[:2]
def change_lowest2(val):
    if val in lowest2:
        return val
    return 20
s.where(s.isin(lowest2), 20)

488 µs ± 703 ns per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [ ]:
s[:10].apply(np.log) # it is appropriate to use .apply

row_1     2.302585
row_2     2.484907
row_3          NaN
row_4     2.397895
row_5     2.302585
row_6     6.907755
row_7     2.484907
row_8     2.564949
row_9     2.302585
row_10         NaN
Name: sample_series, dtype: float64

## indexing

### sampling, head, tails
| Method / Function | Description |
|---|---|
| s.head(n=5) | Return first n rows of the Series. |
| s.tail(n=5) | Return last n rows of the Series. |
| s.sample(n=None, frac=None, replace=False, weights=None, random_state=None, axis=None) | Return a random sample by n or fraction; supports replacement, weights, and seed. |

In [126]:
import pandas as pd
url = 'https://github.com/mattharrison/datasets/raw/master/data/' \
      'vehicles.csv.zip'
df = pd.read_csv(url)
city_mpg = df.city08 # this is identical to df['city08'] 

C:\Users\xiang\AppData\Local\Temp\ipykernel_15592\1492182565.py:4: DtypeWarning: Columns (68,70,71,72,73,74,76,79) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(url)


In [127]:
make=df.make
city3=city_mpg.rename(make.to_dict())

In [66]:
df.columns

Index(['barrels08', 'barrelsA08', 'charge120', 'charge240', 'city08',
       'city08U', 'cityA08', 'cityA08U', 'cityCD', 'cityE', 'cityUF', 'co2',
       'co2A', 'co2TailpipeAGpm', 'co2TailpipeGpm', 'comb08', 'comb08U',
       'combA08', 'combA08U', 'combE', 'combinedCD', 'combinedUF', 'cylinders',
       'displ', 'drive', 'engId', 'eng_dscr', 'feScore', 'fuelCost08',
       'fuelCostA08', 'fuelType', 'fuelType1', 'ghgScore', 'ghgScoreA',
       'highway08', 'highway08U', 'highwayA08', 'highwayA08U', 'highwayCD',
       'highwayE', 'highwayUF', 'hlv', 'hpv', 'id', 'lv2', 'lv4', 'make',
       'model', 'mpgData', 'phevBlended', 'pv2', 'pv4', 'range', 'rangeCity',
       'rangeCityA', 'rangeHwy', 'rangeHwyA', 'trany', 'UCity', 'UCityA',
       'UHighway', 'UHighwayA', 'VClass', 'year', 'youSaveSpend', 'guzzler',
       'trans_dscr', 'tCharger', 'sCharger', 'atvType', 'fuelType2', 'rangeA',
       'evMotor', 'mfrCode', 'c240Dscr', 'charge240b', 'c240bDscr',
       'createdOn', 'modifiedOn

### filter, reindex

| Method / Function | Description |
|---|---|
| s.filter(items=None, like=None, regex=None, axis=None) | Return subset of the Series by exact items list, substring match (like), or regex. |
| s.reindex(index=None, method=None, copy=True, level=None, limit=None, tolerance=None) | Conform Series to new index; supports fill methods (forward/backward), limits, and tolerance for nearest matching. |

In [129]:
city3.filter(like='rd')
# or use regular expression
city3.filter(regex='(Ford)|(Subaru)')

Subaru    17
Subaru    21
Subaru    22
Ford      18
Ford      16
          ..
Subaru    19
Subaru    20
Subaru    18
Subaru    18
Subaru    16
Name: city08, Length: 4256, dtype: int64

### reindex
to select values from a series partially share the same index with another. Unshared index is returned NaN.

| Method / Function | Description |
|---|---|
| s.reindex(index=None, method=None, copy=True, level=None, limit=None, tolerance=None) | Conform Series to new index; supports fill methods (forward/backward), limits, and tolerance for nearest matching. |

In [132]:
s1 = pd.Series([10,20,30], index=['a', 'b', 'c'])
s2 = pd.Series([15,25,35], index=['b', 'c', 'd'])
s2.reindex(s1.index)

a     NaN
b    15.0
c    25.0
dtype: float64

### .iloc[]
| Method / Function | Description |
|---|---|
| s.iloc[idx] | Position-based selection/slicing: scalar, list of positions, integer slice (half-open), boolean mask, or callable. |

In [ ]:
display(city3.iloc[-1])   # scalar
display(city3.iloc[[-1]]) # series
display(city3.iloc[-3:])  
display(city3.iloc[[1,3,5]])
mask4=city3>110 # boolean array
display(city3.iloc[mask4.to_numpy()])

np.int64(16)

Subaru    16
Name: city08, dtype: int64

Subaru    18
Subaru    18
Subaru    16
Name: city08, dtype: int64

Ferrari     9
Dodge      10
Subaru     21
Name: city08, dtype: int64

Mitsubishi    126
Honda         132
smart         122
smart         122
Scion         138
             ... 
Nissan        118
Nissan        114
Tesla         138
Tesla         140
Tesla         115
Name: city08, Length: 87, dtype: int64

### .loc[]
| Method / Function | Description |
|---|---|
| s.loc[idx] | Label-based selection/slicing: scalar, list of labels, label slice (inclusive), boolean mask, Index, or callable. |
| s.sort_index(axis=0, level=None, ascending=True, inplace=False, kind='quicksort', na_position='last', sort_remaining=True, ignore_index=False, key=None) | Return a Series with the index sorted. kind controls algorithm; na_position places NaNs first/last. |


In [97]:
display(city3.head(3))
display(city3.loc['Fisker'])
display(city3.loc[['Fisker']]) # as a series
display(city3.loc['Subaru'])   # return multiple ones
display(city3.loc[['Fisker','Lamborghini']]) # use list notation

Alfa Romeo    19
Ferrari        9
Dodge         23
Name: city08, dtype: int64

np.int64(20)

Fisker    20
Name: city08, dtype: int64

Subaru    17
Subaru    21
Subaru    22
Subaru    19
Subaru    20
          ..
Subaru    19
Subaru    20
Subaru    18
Subaru    18
Subaru    16
Name: city08, Length: 885, dtype: int64

Fisker         20
Lamborghini     8
Lamborghini     8
Lamborghini     8
Lamborghini     8
               ..
Lamborghini     6
Lamborghini     8
Lamborghini     8
Lamborghini     8
Lamborghini     8
Name: city08, Length: 129, dtype: int64

In [103]:
display(city3.sort_index()['Fisker':'Lamborghini']) # once sorted, can access the range 
display(city3.sort_index()['F':'L']) # including exactly with 'Lamb' but not start with Lamb!

Fisker         20
Ford           18
Ford           21
Ford           13
Ford           17
               ..
Lamborghini    12
Lamborghini     9
Lamborghini     8
Lamborghini    13
Lamborghini     8
Name: city08, Length: 10910, dtype: int64

Federal Coach    15
Federal Coach    13
Federal Coach    13
Federal Coach    14
Federal Coach    13
                 ..
Kia              25
Kia              21
Kia              15
Kia              19
Koenigsegg       11
Name: city08, Length: 11093, dtype: int64

> Note: `.loc[]` follows a closed-interval

In [ ]:
idx=pd.Index(['Fisker', 'Fisker']) # can duplicate the values with the same index
city3.loc[idx]

Fisker    20
Fisker    20
Name: city08, dtype: int64

In [ ]:
mask3=city3>100 # boolean array
city3.loc[mask3] # only returns values where mask3 is true

MINI          102
Nissan        106
Mitsubishi    126
Nissan        106
BMW           107
             ... 
Nissan        114
Tesla         138
Tesla         140
Tesla         115
Tesla         104
Name: city08, Length: 110, dtype: int64

In [117]:
display(city3.mul(0.9).loc[lambda s_: s_>s_.mean()].tail(3))
mvalue=city3.mul(0.9).mean()
display(city3.mul(0.9).loc[lambda s_: s_>mvalue].tail(3))
display('------------')
display(city3.mul(0.9).loc[lambda s_: s_>city3.mean()].tail(3))

Saturn    18.9
Subaru    17.1
Subaru    18.0
Name: city08, dtype: float64

Saturn    18.9
Subaru    17.1
Subaru    18.0
Name: city08, dtype: float64

'------------'

Saturn    18.9
Saturn    21.6
Saturn    18.9
Name: city08, dtype: float64

> Note: using lambda function to filter series after an intermediate step (aka `s_`). 

### reset idx
| Method / Function | Description |
|---|---|
| s.reset_index(level=None, drop=False, name=None, inplace=False) | Return a DataFrame with a new integer index and the old index moved to a column. |
| s.reset_index(drop=True) | Return Series with a new integer index and the old index is deleted. |

In [88]:
city_mpg2=city_mpg.rename(make.to_dict())
display(city_mpg2.reset_index().head(3)) # returns a DataFrame old index becomes a new column called index
display(city_mpg2.reset_index(drop=True).head(2)) # returns a series

,index,city08
0,Alfa Romeo,19
1,Ferrari,9
2,Dodge,23


0    19
1     9
Name: city08, dtype: int64

### renaming index

| Method / Function | Description |
|---|---|
| s.rename(index=None, func, dict, series, \*, level=None, errors='ignore') | 1) Return a Series with updated 1 index mapping for `index=<1 key dict>`; 2) update `.name` if index is scalar; 3) `index=<func, Series, or dict>` to map old->new. |
| s.index | Returns the Index object of the Series. |

In [78]:
from IPython.display import display
display(city_mpg.head())
display(make.head())
dict(list(make.to_dict().items())[:5] ) 
#       make.to_dict()    to dictionary
#               .items()  returns both key and value of
# list(                 ) key and value pairs to tuple and then stacked into a list

0    19
1     9
2    23
3    10
4    17
Name: city08, dtype: int64

0    Alfa Romeo
1       Ferrari
2         Dodge
3         Dodge
4        Subaru
Name: make, dtype: object

{0: 'Alfa Romeo', 1: 'Ferrari', 2: 'Dodge', 3: 'Dodge', 4: 'Subaru'}

In [79]:
list(make.items())[:5]

[(0, 'Alfa Romeo'), (1, 'Ferrari'), (2, 'Dodge'), (3, 'Dodge'), (4, 'Subaru')]

In [ ]:
city_mpg.rename(index={0:'first'}).head() # change values of particular inices

first    19
1         9
2        23
3        10
4        17
Name: city08, dtype: int64

In [84]:
display(city_mpg.rename(index='city08mpg').name)  # only changes the name of the series
display(city_mpg.rename(index='city08mpg').head(3))


'city08mpg'

0    19
1     9
2    23
Name: city08mpg, dtype: int64

In [86]:
display(city_mpg.rename(make.to_dict()).head(2))
def str_index(val):
    return f'idx-{val}'
display(city_mpg.rename(str_index).head(4))



Alfa Romeo    19
Ferrari        9
Name: city08, dtype: int64

idx-0    19
idx-1     9
idx-2    23
idx-3    10
Name: city08, dtype: int64

## string operation
| Method | Description |
|---|---|
| `.str.capitalize()` | Capitalize strings |
| `.str.casefold()` | Lowercase Unicode / caseless strings. |
| `.str.cat(others=None, sep='', na_rep=None, join='inner')` | If `others` is None, return a single string with values separated by `sep`. Otherwise align the index (if `others` is a Series) and concatenate values. |
| `.str.center(width, fillchar=' ')` | Center align strings (pad with `fillchar`). |
| `.str.contains(pat, case=True, flags=0, na=np.nan, regex=True)` | Return a boolean array indicating whether `pat` matches each value. |
| `.str.count(pat, flags=0)` | Return a Series with the count of how many times `pat` occurs in each value. |
| `.str.decode(encoding)` | Decode bytestrings to Unicode strings (works on bytes). |
| `.str.encode(encoding)` | Encode Unicode strings to bytestrings. |
| `.str.endswith(pat, na=np.nan)` | Return boolean array indicating whether values end with `pat`. |
| `.str.extract(pat, flags=0, expand=True)` | Return a DataFrame with the first match from each regex capture group in its own column (use named groups for column names). Returns a Series if `expand=False`. |
| `.str.extractall(pat, flags=0)` | Return a DataFrame with all matches; each capture group in its own column. Result has a MultiIndex where inner index `match` shows match number. |
| `.str.find(sub, start=None, end=None)` | Return the lowest index of `sub` in each value; `-1` if not found. |
| `.str.findall(pat, flags=0)` | Return a Series of lists with all regex matches for each value. |
| `.str.get(i)` | Return a Series with the result of `val[i]` for each value `val` in the Series. |
| `.str.get_dummies(sep='&#124;')` | Return a DataFrame with one-hot columns for each value; 0/1 indicates absence/presence. If a string contains multiple values, separate them with `sep`. |
| `.str.index(sub, start=None, end=None)` | Return the lowest index of `sub`; raises `ValueError` if not found. |
| `.str.isalnum()` | Return boolean array indicating if characters are alphanumeric. |
| `.str.isalpha()` | Return boolean array indicating if characters are alphabetic. |
| `.str.isdecimal()` | Return boolean array indicating if characters are decimal. |
| `.str.isdigit()` | Return boolean array indicating if characters are digits. |
| `.str.islower()` | Return boolean array indicating if characters are lowercase. |
| `.str.isnumeric()` | Return boolean array indicating if characters are numeric. |
| `.str.isspace()` | Return boolean array indicating if characters are whitespace. |
| `.str.istitle()` | Return boolean array indicating if characters are titlecase. |
| `.str.isupper()` | Return boolean array indicating if characters are uppercase. |
| `.str.join(sep)` | Given a Series of lists of strings, join each list element with `sep`. |
| `.str.len()` | Return a Series with the length of each value (works with lists/collections). |
| `.str.ljust(width, fill=' ')` | Return a left-justified Series padded to `width` with `fill`. |
| `.str.lower()` | Return a lowercase Series. |
| `.str.lstrip(to_strip=None)` | Return a Series with left-stripped `to_strip` (whitespace default). |
| `.str.match(pat, case=True, flags=0, na=np.nan)` | Return boolean array indicating whether `pat` matches values (anchored at the beginning). Use `.str.contains` to match anywhere. |
| `.str.normalize(form)` | Return Unicode normalized form for the Series (`'NFC'`, `'NFKC'`, `'NFD'`, or `'NFKD'`). |
| `.str.pad(width, side='left', fill=' ')` | Return a padded Series of length `width`. `side` can be `'left'`, `'right'`, or `'both'`. |
| `.str.partition(sep, expand=True)` | Return a DataFrame with three columns: part before first `sep`, the `sep`, and part after. |
| `.str.repeat(repeats)` | Return a Series with values repeated `repeats` times (scalar or list). |
| `.str.replace(pat, repl, n=-1, case=True, flags=0, regex=True)` | Return a Series with `pat` replaced by `repl`. `n` limits replacements. `repl` may be a string or a callable receiving a match object. |
| `.str.rfind(sub, start=None, end=None)` | Return the highest index of `sub`; `-1` if not found. |
| `.str.rindex(sub, start=None, end=None)` | Return the highest index of `sub`; raises `ValueError` if not found. |
| `.str.rjust(width, fill=' ')` | Return a right-justified Series padded to `width` with `fill`. |
| `.str.rpartition(sep, expand=True)` | Return a DataFrame with three columns: part before last `sep`, the `sep`, and part after. |
| `.str.rsplit(pat, n=-1, expand=False)` | Return a Series (if `expand=False`) with lists of values split from the right, limited to `n` splits. |
| `.str.rstrip(to_strip=None)` | Return a Series with right-stripped `to_strip` (whitespace default). |
| `.str.slice(start=None, stop=None, step=None)` | Return a Series equivalent to `s[start:stop:step]`. |
| `.str.slice_replace(start=None, stop=None, repl=None)` | Return a Series with the slice replaced by `repl`. |
| `.str.split(pat, n=-1, expand=False)` | Return a Series (if `expand=False`) with lists of values split by `pat`, limited to `n` splits. |
| `.str.startswith(pat, na=np.nan)` | Return boolean array indicating whether values start with `pat`. |
| `.str.strip(to_strip=None)` | Return a Series with left and right stripped `to_strip` (whitespace default). |
| `.str.swapcase()` | Return a Series with characters swapped case. |
| `.str.title()` | Return a titlecase Series. |
| `.str.translate(table)` | Return a Series using a dictionary `table` to replace characters. `table` maps code points (integers) to new code points; keys mapped to `None` are deleted. |
| `.str.upper()` | Return an uppercase Series. |
| `.str.wrap(width)` | Return a line-wrapped Series limited to `width`. |
| `.str.zfill(width)` | Return a Series left-padded with `'0'` to `width`. |

In [13]:
import numpy as np
import pandas as pd
age = pd.Series(['0-10', '11-15', '11-15', '61-65', '46-50'])
age.str.split('-', expand=True)\
    .astype(int)\
    .pipe(lambda df_: pd.Series(np.random.randint(df_.iloc[:,0], df_.iloc[:,1]), index=df_.index))


0     9
1    13
2    13
3    63
4    47
dtype: int32

> Note: using `.pipe()` to pass the entire series as argument to the random generator; `.apply()` operate one row at a time.

## categorical manipulation
| Method | Description |
|---|---|
| `.astype(dtype)` | Return a Series converted to categories. Set `dtype` to `'category'` for an unordered category or use `CategoricalDtype` for an ordered category. |
| `pd.CategoricalDtype(categories, ordered=False)` | Create a categorical dtype. Set `categories` to a list of category values; `ordered=True` makes it ordinal. |
| `pd.cut(x, bins, right=True, labels=None, retbins=False, precision=3, include_lowest=False, duplicates='raise', ordered=True)` | Bin values from `x` (a Series). If `bins` is an integer, use equal-width bins. If `bins` is a list of numbers, use those as edges. `right` controls interval closure. `labels` names the bins. Out-of-bounds values become missing. |
| `pd.qcut(x, q, labels=None, retbins=False, precision=3, duplicates='raise')` | Bin values from `x` into `q` equal-sized quantile bins (e.g., `q=4` for quartiles). Alternatively pass a list of quantile edges. Out-of-bounds values become missing. |
| `.cat.add_categories(new_categories)` | Return a Series with `new_categories` added to the categorical dtype (new categories appended at the end for ordered categories). |
| `.cat.as_ordered()` | Convert a categorical Series to an ordered (ordinal) Series. Use `.reorder_categories` or `CategoricalDtype` to specify the order. |
| `.cat.categories` | Property returning the Index of categories. |
| `.cat.codes` | Property returning a Series of integer codes (indices into the categories) for each value. |
| `.cat.ordered` | Boolean property indicating whether the Series is ordered. |
| `.cat.remove_categories(removals)` | Return a Series with the specified categories removed; any values in those categories become `NaN`. |
| `.cat.remove_unused_categories()` | Return a Series with categories that are not used removed from the dtype. |
| `.cat.rename_categories(new_categories)` | Return a Series with categories renamed; `new_categories` can be a list (replacement) or a dict mapping old→new. |
| `.cat.reorder_categories(new_categories)` | Return a Series with categories replaced by the provided list `new_categories` (useful to set a specific ordering). |

In [25]:
import pandas as pd
url = 'https://github.com/mattharrison/datasets/raw/master/' \
    'data/vehicles.csv.zip'
# df0 = pd.read_csv(url, engine='pyarrow', dtype_backend='pyarrow') 
df0 = pd.read_csv(url, dtype='str') 
make = df0.make


In [26]:
make.value_counts() # counting frequencies of different categories

make
Chevrolet                           4003
Ford                                3371
Dodge                               2583
GMC                                 2494
Toyota                              2071
                                    ... 
RUF Automobile                         1
General Motors                         1
Environmental Rsch and Devp Corp       1
Goldacre                               1
Isis Imports Ltd                       1
Name: count, Length: 136, dtype: int64

In [27]:
make.astype('category')
make.str.upper()

0        ALFA ROMEO
1           FERRARI
2             DODGE
3             DODGE
4            SUBARU
            ...    
41139        SUBARU
41140        SUBARU
41141        SUBARU
41142        SUBARU
41143        SUBARU
Name: make, Length: 41144, dtype: object

> converting as categorical type saves memory and speed up operation!

### ordinal categories

In [30]:
from IPython.display import display

make_type = pd.CategoricalDtype(categories=sorted(make.unique()), ordered=True)
ordered_make=make.astype(make_type)
display(ordered_make.head())
display(len(ordered_make[ordered_make>'AM General']))

0    Alfa Romeo
1       Ferrari
2         Dodge
3         Dodge
4        Subaru
Name: make, dtype: category
Categories (136, object): ['AM General' < 'ASC Incorporated' < 'Acura' < 'Alfa Romeo' ... 'Volvo' < 'Wallace Environmental' < 'Yugo' < 'smart']

41138

In [ ]:
# way 2 return the alphabetical order
ordered_make2=make.astype('category').cat.as_ordered()
display(ordered_make2)
display(len(ordered_make[ordered_make2>'AM General']))

0        Alfa Romeo
1           Ferrari
2             Dodge
3             Dodge
4            Subaru
            ...    
41139        Subaru
41140        Subaru
41141        Subaru
41142        Subaru
41143        Subaru
Name: make, Length: 41144, dtype: category
Categories (136, object): ['AM General' < 'ASC Incorporated' < 'Acura' < 'Alfa Romeo' ... 'Volvo' < 'Wallace Environmental' < 'Yugo' < 'smart']

41138

### .cat accessor
> Need a list with the same length as the current categories or a dictionary mapping old values to new values

In [32]:
ordered_make2.cat.rename_categories([c.lower() for c in ordered_make2.cat.categories])

0        alfa romeo
1           ferrari
2             dodge
3             dodge
4            subaru
            ...    
41139        subaru
41140        subaru
41141        subaru
41142        subaru
41143        subaru
Name: make, Length: 41144, dtype: category
Categories (136, object): ['am general' < 'asc incorporated' < 'acura' < 'alfa romeo' ... 'volvo' < 'wallace environmental' < 'yugo' < 'smart']

In [33]:
ordered_make.cat.rename_categories({c:c.lower() for c in ordered_make.cat.categories})

0        alfa romeo
1           ferrari
2             dodge
3             dodge
4            subaru
            ...    
41139        subaru
41140        subaru
41141        subaru
41142        subaru
41143        subaru
Name: make, Length: 41144, dtype: category
Categories (136, object): ['am general' < 'asc incorporated' < 'acura' < 'alfa romeo' ... 'volvo' < 'wallace environmental' < 'yugo' < 'smart']

### gotchas

Applying the `.value_counts` method or `.groupby` to categorical data uses all categories even if they have no values.

In [ ]:
gotchas=pd.Series(['a', 'b', 'a', 'c', 'd', 's']).astype('category').cat.as_ordered()
display(gotchas)
display(gotchas[[0]].value_counts()) # count all categories

0    a
1    b
2    a
3    c
4    d
5    s
dtype: category
Categories (5, object): ['a' < 'b' < 'c' < 'd' < 's']

a    1
b    0
c    0
d    0
s    0
Name: count, dtype: int64

In [ ]:
gotchas.head(1).groupby(gotchas.head(1), observed=False).first() 

a      a
b    NaN
c    NaN
d    NaN
s    NaN
dtype: category
Categories (5, object): ['a' < 'b' < 'c' < 'd' < 's']

In [ ]:
gotchas.head(1).groupby(gotchas.head(1), observed=True).first() 
# observed=T tells to only include results for which there are values

a    a
dtype: category
Categories (5, object): ['a' < 'b' < 'c' < 'd' < 's']

**exercise**: new categories

Suppose I want country from make, but I only want US and German categories, and I want to label everything else as “Other”

In [61]:
newcat={'Ford': 'US', 'Tesla': 'US',
        'Chevrolet': 'US', 'Dodge': 'US',
        'Oldsmobile': 'US', 'Plymouth': 'US',
        'BMW': 'German'}
res=pd.Series(['Ford','Chevrolet','Tesla','Dodge','Oldsmobile','Plymouth','BMW',"Toyota", 'Smart'])

In [ ]:
# way 1
seen = None
res0=res
for old, new in newcat.items():
    mask = res0.str.contains(old)
    if seen is None:
        seen = mask
    else:
        seen |= mask
    res = res.where(~mask, new) # keep true and replace false with new
res = res.where(seen, "Other")

display(res)

0        US
1        US
2        US
3        US
4        US
5        US
6    German
7     Other
8     Other
dtype: object

In [48]:
# way 2: us np.select
res2 = ordered_make.astype(str)
condlist=[
        res2.eq('Ford'),        # case 1
        res2.eq('Chevrolet'),   # case 2
        res2.eq('Tesla'),
        res2.eq('Dodge'),       # case 4
        res2.eq('Oldsmobile'),  # case 5
        res2.eq('Plymouth'),
        res2.eq('BMW')
        # add more cases here...
    ]

choicelist=[
        'US',   # result for case 1
        'US',   # result for case 2
        'US',   # result for case 3
        'US',   # result for case 4
        'US',   # result for case 5
        'US',   # result for case 6
        'German',   # result for case 7
        # corresponding results...
    ]

res3=pd.Series(np.select(condlist, choicelist,default='Other')).astype('category')
display(res3)


0        Other
1        Other
2           US
3           US
4        Other
         ...  
41139    Other
41140    Other
41141    Other
41142    Other
41143    Other
Length: 41144, dtype: category
Categories (3, object): ['German', 'Other', 'US']